The importance of a  recursive forecast consists on the model using its own predictions to predict further more into the future

In [2]:
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error

# Load data
df = pd.read_csv("/Users/jussaragaspar/Downloads/Final project/time_series_60min_singleindex_filtered.csv")

# Convert timestamp
df["utc_timestamp"] = pd.to_datetime(df["utc_timestamp"])
df = df.set_index("utc_timestamp")

# Keep only FRANCE demand first
df_model = df[["FR_load_actual_entsoe_transparency"]].dropna().copy()

# Create lagged features
df_model["lag_1"] = df_model["FR_load_actual_entsoe_transparency"].shift(1)
df_model["lag_24"] = df_model["FR_load_actual_entsoe_transparency"].shift(24)
df_model["lag_168"] = df_model["FR_load_actual_entsoe_transparency"].shift(168)

# Time features
df_model["hour"] = df_model.index.hour
df_model["dayofweek"] = df_model.index.dayofweek

df_model = df_model.dropna()

X = df_model[["lag_1", "lag_24", "lag_168", "hour", "dayofweek"]]
y = df_model["FR_load_actual_entsoe_transparency"]

In [3]:
df["hour"] = df.index.hour
df["dayofweek"] = df.index.dayofweek
df["month"] = df.index.month
df["is_weekend"] = (df.index.dayofweek >= 5).astype(int)

In [4]:
X = df_model[["lag_1", "lag_24", "lag_168", "hour", "dayofweek"]]
y = df_model["FR_load_actual_entsoe_transparency"]

In [5]:
# Train-test split
X_train, X_test = X.iloc[:int(len(X) * 0.8)], X.iloc[int(len(X) * 0.8):]
y_train, y_test = y.iloc[:int(len(y) * 0.8)], y.iloc[int(len(y) * 0.8):]

In [6]:
from sklearn.ensemble import RandomForestRegressor

model = RandomForestRegressor(n_estimators=100, random_state=42)
model.fit(X_train, y_train)

RandomForestRegressor(random_state=42)

In [9]:
import pandas as pd
import numpy as np

from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error

# ==========================================
# TARGET
# ==========================================
target_col = "FR_load_actual_entsoe_transparency"

df.index = pd.to_datetime(df.index)
series = df[target_col].dropna()

# ==========================================
# FUNCTION
# ==========================================
def create_lag_features(series, n_lags):
    X, y = [], []
    
    values = series.values
    
    for i in range(n_lags, len(values)):
        X.append(values[i - n_lags:i])
        y.append(values[i])
        
    return np.array(X), np.array(y)

# ==========================================
# LAG VALUES TO TEST
# ==========================================
lag_list = [1, 48, 72, 168]

results = []

# ==========================================
# LOOP
# ==========================================
for n_lags in lag_list:
    print(f"\nTesting lag = {n_lags}")
    
    X, y = create_lag_features(series, n_lags)
    
    split_idx = int(len(X) * 0.8)
    
    X_train, X_test = X[:split_idx], X[split_idx:]
    y_train, y_test = y[:split_idx], y[split_idx:]
    
    model = RandomForestRegressor(
        n_estimators=150,
        max_depth=12,
        random_state=42,
        n_jobs=-1
    )
    
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    
    mae = mean_absolute_error(y_test, y_pred)
    rmse = np.sqrt(mean_squared_error(y_test, y_pred))
    
    print(f"MAE: {mae:.2f}, RMSE: {rmse:.2f}")
    
    results.append((n_lags, mae, rmse))

# ==========================================
# RESULTS TABLE
# ==========================================
results_df = pd.DataFrame(results, columns=["lags", "MAE", "RMSE"])

print("\nFinal comparison:")
print(results_df.sort_values("MAE"))


Testing lag = 1
MAE: 1747.95, RMSE: 2431.68

Testing lag = 48
MAE: 581.87, RMSE: 1405.75

Testing lag = 72
MAE: 587.33, RMSE: 1413.12

Testing lag = 168
MAE: 630.26, RMSE: 1444.48

Final comparison:
   lags          MAE         RMSE
1    48   581.865181  1405.750533
2    72   587.331636  1413.115909
3   168   630.264367  1444.484275
0     1  1747.946632  2431.676511




# ⚡ Forecasting Model and Recursive Logic — Full Explanation

The objective of this model is to **predict future electricity demand and simulate how the power grid will behave**. This is a key component of the digital twin, because it allows us to move from analyzing past data to **simulating future conditions**.

The process starts with preparing the data so that the model can learn how electricity demand behaves over time. . These include lag variables such as the demand one hour ago (`lag_48`), the demand at the same hour the previous day (`lag_24`), and the demand at the same hour the previous week (`lag_168`). I've also includd time-related features such as the hour of the day and the day of the week. These features are essential because electricity demand follows strong daily and weekly patterns, and the model needs this information to learn those patterns.

Once the data is prepared, the model is trained using historical data. During training, the model learns the relationship between past demand and future demand. For example, it learns that demand is usually higher in the morning and evening, lower at night, and different on weekends compared to weekdays.

After training, the model can be used to make predictions. First, it predicts the demand for the next hour. However, predicting only one step ahead is not sufficient for a digital twin. We need to simulate multiple future time steps, such as the next 24 hours. To achieve this, I used a method called **recursive forecasting**.

Recursive forecasting works by predicting one step at a time and then using that prediction to generate the next one. In each iteration of the loop, the model takes the most recent data available, including previous predictions, and predicts the next hour. The process follows several steps: first, the model retrieves the most recent demand values (such as the last hour, the previous day, and the previous week). Then, it advances the time by one hour and extracts the corresponding time features (hour and day of the week). These inputs are used to build a new input vector, which is passed to the model to generate a prediction. This prediction is then stored and added to the dataset, effectively becoming part of the “history” used for the next prediction.

This process is repeated multiple times (for example, 24 times) to simulate the next 24 hours. As a result, the model generates a full sequence of future demand values. This approach is necessary because real future data is unknown, so the model must rely on its own predictions to build the future step by step.

Once the future demand is predicted, it is combined with the demand from other countries to represent the total system demand. At the same time, electricity generation is estimated using available data such as renewable generation (wind and solar) and a base generation component that represents other energy sources like nuclear or gas. The difference between total generation and total demand is then calculated to obtain the **grid balance**, which indicates whether the system has enough electricity.

The grid balance is the core indicator of the system. If it is positive, there is enough electricity; if it is negative, there is a shortage. This allows us to evaluate the stability of the system over time.

Finally, this forecasting framework enables the creation of scenarios. By modifying demand or generation (for example, increasing demand or reducing wind production), we can simulate different situations and observe how the grid reacts. This is what makes the model a true digital twin: it not only predicts the future but also allows us to test how the system behaves under different conditions.

One limitation of this approach is that errors can accumulate over time, because each prediction is used as input for the next one. However, this is a known characteristic of recursive forecasting and is acceptable for short-term simulations.

In short, the forecasting model uses past data to learn patterns, predicts future demand step by step using a recursive approach, and enables the simulation of the power grid under different scenarios. This transforms the project from simple data analysis into a dynamic system capable of evaluating future performance and stability.
